# Entraînement du classifieur d'espèces — Fisher Link (Gabon)

Ce notebook entraîne le modèle utilisé par `server/fishid.js`, sur Google Colab avec un GPU gratuit. Il **remplace** le premier modèle générique (9 espèces du dataset Kaggle, non pertinentes pour le Gabon — bar, sprat, truite...) par un modèle scopé aux espèces réellement pêchées au Gabon (pêche artisanale côtière, Port-Gentil) — voir `planning/discrepancies.md` § Species Scope — revised to Gabon-specific.

**Avant de commencer :** Menu **Exécution → Modifier le type d'exécution → GPU (T4)**.

Espèces couvertes (12) : Ethmalose, Otolithe sénégalais/Courbine, Mâchoiron, Capitaine, Carpe rouge/Pagre, Bar barracuda, Thiof/Mérou, Carangue, Sole, Thon, Crevette, Sardinelle.

Données : téléchargées automatiquement depuis GBIF (occurrences avec photo, filtrées sur l'Afrique) via `ml/download_gbif.py` — pas de clé API requise, contrairement au dataset Kaggle du modèle précédent. **Licence : majoritairement CC BY-NC (usage non-commercial)** — acceptée pour l'instant, à revoir avant toute commercialisation de FisherLink (voir `planning/project_log.md` #14). Chaque photo est attribuée (auteur/licence/source) dans un `attribution.json` par espèce, téléchargé à la fin avec le modèle — à conserver, ne pas perdre.

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠ Pas de GPU détecté — vérifiez Exécution > Modifier le type d'exécution > GPU")

## 1. Récupérer les scripts du projet (clone repo)

On clone le dépôt GitHub pour réutiliser exactement les mêmes scripts que le dossier `ml/` local (`download_gbif.py`, `train.py`, `export_onnx.py`) — pas de duplication de code.

In [ ]:
!rm -rf /content/repo
!git clone --depth 1 https://github.com/KingjulianB/ha-addon-peche.git /content/repo
%cd /content/repo/peche/ml
!pip install -q onnx onnxscript

## 2. Télécharger les photos d'entraînement (GBIF)

Aucune clé API nécessaire — l'API GBIF est publique. `--max-per-species 150` garde un premier jeu de données raisonnable à entraîner (certaines espèces ont moins de photos disponibles, ex. Otolithe sénégalais ~23, Carangue ~46 — normal, voir `planning/discrepancies.md`).

In [ ]:
!python download_gbif.py --out /content/data --max-per-species 150

## 3. Entraîner

`torch`/`torchvision` sont déjà installés sur Colab. 10 époques est un point de départ raisonnable pour un premier modèle — augmentez si la précision de validation continue de progresser. Avec des classes aussi fines que 23-46 images (Otolithe sénégalais, Carangue), ne soyez pas surpris par une précision de validation modeste au départ : ce n'est qu'un premier passage, à affiner ensuite avec vos propres photos de prises réelles (voir `ml/README.md` § Réentraînement).

In [ ]:
!python train.py --data-dir /content/data --epochs 10 --out /content/model.pt --labels /content/labels.json

## 4. Exporter au format ONNX

In [ ]:
!python export_onnx.py --model /content/model.pt --labels /content/labels.json --out /content/model.onnx

## 5. Télécharger le modèle

Placez les fichiers téléchargés dans `ha-addon/peche/ml/` en local (à côté de `species_weights.json`, qu'il faudra aussi mettre à jour avec les 12 nouvelles clés d'espèces — voir `ml/README.md`), redémarrez le serveur : `fishIdConfigured()` deviendra automatiquement vrai. Le zip `attributions.zip` (licence/auteur/source de chaque photo CC BY-NC utilisée) est aussi téléchargé — **à conserver comme trace, ne pas supprimer** (voir `planning/project_log.md` #14).

In [ ]:
import os
import zipfile

with zipfile.ZipFile("/content/attributions.zip", "w") as zf:
    for root, _, filenames in os.walk("/content/data"):
        for fn in filenames:
            if fn == "attribution.json":
                full = os.path.join(root, fn)
                arcname = os.path.relpath(full, "/content/data")
                zf.write(full, arcname)

from google.colab import files

files.download("/content/model.onnx")
files.download("/content/labels.json")
files.download("/content/attributions.zip")

# Défense en profondeur : si un futur torch réexternalise les poids malgré
# dynamo=False, on ne veut pas re-rater silencieusement ce fichier.
data_file = "/content/model.onnx.data"
if os.path.exists(data_file):
    print("⚠ Fichier de poids externe détecté, téléchargement aussi :", data_file)
    files.download(data_file)